# SafePath — Edge Safety Score & Routing

---

## Context — How Each Score Was Built

Before this notebook runs, four feature scores were computed and attached to
every street edge in the San Diego OSM walking network. Each score is a number
between 0 and 1 where **higher = safer or more comfortable** for a pedestrian.

| Score | Range | How it was computed | Source data |
|---|---|---|---|
| `crime_score_day` | 0–1 | SDPD incidents within 100 m of the edge. Each incident has a `severity_score` encoding how serious the crime is and how directly it threatens a pedestrian. Scores are summed, log-normalized, and flipped so 1 = no crime. Daytime incidents only (6 am–8 pm). | SDPD calls for service 2023–2025 |
| `crime_score_night` | 0–1 | Same as above but nighttime incidents only (9 pm–5 am). | SDPD calls for service 2023–2025 |
| `light_score` | 0–1 | Count of active City of San Diego streetlights within 30 m of the edge, normalized by the highest count across all edges. Higher = more streetlights = better lit. 99.99% of edges have a real score. | SD City streetlight inventory |
| `walk_score` | 0–1 | EPA National Walkability Index (`NatWalkInd`, 1–20) for the census block group the edge midpoint falls in, normalized to 0–1. Measures destination density and foot traffic. ~35% of edges use 0.5 fallback (ocean, parks, military land). | EPA Walkability Index |
| `road_class_score` | 0–1 | OSM `highway` tag mapped to a fixed comfort score. Footway/pedestrian = 1.0, residential = 0.70, tertiary = 0.50, secondary = 0.35, primary = 0.20, motorway = 0.0. Unlisted tags = 0.5. | OpenStreetMap |

---

## Data Required

This notebook needs two files in `data/processed/`:

| File | What it contains |
|---|---|
| `data/processed/sd_walk_graph.graphml` | San Diego OSM walking network — nodes are intersections, edges are street segments with `length`, `name`, `highway`, and geometry |
| `data/processed/edge_scores.csv` | One row per edge `(u, v, key)` with the five score columns above |

Both files are on Google Drive. Download and place them in `data/processed/` before running.

---

## Goal Of This Notebook

**Step 1 — Safety score per edge.**
Combine the five feature scores into a single `safety_score` (0–1) using a
weighted average split by time of day:

```
Daytime  (6 am–8 pm):  crime=0.40, light=0.25, walk=0.20, road=0.15
Nighttime (9 pm–5 am): crime=0.35, light=0.35, walk=0.15, road=0.15
```

**Step 2 — Safety cost per edge.**
Convert `safety_score` into a routing cost NetworkX can minimize:

```
safety_cost = length × (1 + 8 × (1 − safety_score))
```

**Step 3 — Three routes.**
For any origin–destination pair, compute fastest, safest, and balanced.

---

## Understanding The Formulas — Read Before Running

### Why Safety Cost Is Calculated This Way

NetworkX finds the path that minimizes total cost. We can't feed it `safety_score`
directly because a higher score means *safer* — and NetworkX would then seek out
the most dangerous streets. The formula solves this in three steps:

**1. Flip the score:**
`(1 − safety_score)` converts the score so 0 = perfectly safe, 1 = maximally
dangerous. Now NetworkX is minimizing in the right direction.

**2. Amplify the difference:**
Without any multiplier, a safe edge (score 0.9) and a dangerous one (score 0.1)
would cost 1.1× and 1.9× their length — a difference too small to justify a detour.
Multiplying by 8 widens that gap to 1.8× vs 8.2× — now the router has a genuine
reason to go around the dangerous street.

**3. Keep physical distance meaningful:**
Adding 1 before multiplying by length ensures a perfectly safe edge (score=1.0)
still costs exactly its real physical length. Without this, the formula would
give safe streets a cost of 0 — the router could treat safe streets as free.

The result:

| `safety_score` | `safety_cost` |
|---|---|
| 1.0 — perfect | 1× length |
| 0.75 | 3× length |
| 0.5 — average | 5× length |
| 0.25 | 7× length |
| 0.0 — worst | 9× length |

---

### Why 8 Might Not Be The Right Multiplier

The multiplier of 8 is a **starting point, not a final answer**. It was chosen
based on early testing with the score distribution. Whether it is right depends
on how spread out the safety scores actually are across the network.

**If the multiplier is too low (e.g. 3–4):**
The cost difference between a safe and a dangerous edge stays small. The router
will barely deviate from the fastest route — the safety signal is drowned out
by physical distance. The safest and fastest routes will look nearly identical
on the map.

**If the multiplier is too high (e.g. 14–16):**
The router aggressively avoids even mildly lower-scoring streets. The safest
route can end up 2–3× longer than the fastest. A pedestrian who is offered a
route that adds 20 minutes to avoid a block that is only slightly less safe
is unlikely to follow it — the app loses credibility.

**How to judge if 8 is right for your data:**
Run the fastest and safest routes on a few test pairs. If the routes are
identical, raise the multiplier. If safest is more than 25–30% longer than
fastest, lower it. The right value is where the safest route takes a clear
but reasonable detour around genuinely dangerous streets.

---

### Why The Balanced Route Is Calculated This Way

```
balanced_cost = alpha × length + (1 − alpha) × safety_cost
```

The formula is a linear blend between two extremes:

| alpha | What happens |
|---|---|
| 1.0 | Formula collapses to just `length` — identical to fastest |
| 0.5 | Both contribute equally — current default |
| 0.0 | Formula collapses to just `safety_cost` — identical to safest |

This formula is intentionally simple. It is easy to explain: alpha is a dial
between speed and safety. A user who wants mostly safe but not too long tries
alpha=0.3. A user who just wants a slight improvement tries alpha=0.7.

**This is a suggestion, not a fixed design.** Some alternatives worth trying:

- `sqrt(length × safety_cost)` — geometric mean, penalizes routes that are
  *both* long and dangerous more aggressively
- Use a lower multiplier for balanced than for safest (e.g. 4 for balanced,
  8 for safest) — this naturally produces a gentler middle route without
  needing alpha at all
- Weighted crime-only avoidance for balanced and full score for safest

---

### What Changing These Numbers Does To The Path

| Change | Effect on routing | When to try it |
|---|---|---|
| Raise safety_cost multiplier | Router avoids dangerous streets more aggressively, adds more walking distance | Routes barely differ from fastest |
| Lower safety_cost multiplier | Router stays closer to the direct path, less sensitive to score differences | Safest route is unreasonably long |
| Lower alpha (toward 0) | Balanced behaves more like safest — safety dominates | Users who prioritize safety over time |
| Raise alpha (toward 1) | Balanced behaves more like fastest — barely deviates | Users who want a slight safety improvement only |
| Both high multiplier and low alpha | Extreme safety focus — path may go well out of its way | Testing the outer limits of the scoring |
| Both low multiplier and high alpha | Almost identical to fastest — scoring barely influences the route | Checking if scores are differentiated enough |

The multiplier and alpha interact: a high multiplier with alpha=0.5 often
produces the same route as a lower multiplier with alpha=0.0. Start with
multiplier=8 and alpha=0.5, then move one knob at a time and observe what
changes before adjusting both together.


In [ ]:
import os
import json

import numpy as np
import pandas as pd
import geopandas as gpd

import osmnx as ox                          # load/save graphml, nearest_nodes
import networkx as nx                       # shortest_path, graph traversal

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D         # custom legend entries for route maps
from shapely.geometry import LineString     # fallback edge geometry for plotting
from datetime import datetime               # system clock for day/night detection
